Filters for Poitroit Filter Bank

In [11]:
import numpy as np
from scipy.io import wavfile
from IPython.display import Audio
import sounddevice as sd
import ipywidgets as widgets
from IPython.display import display


In [34]:
    fs = 48000.0
    f0 = 200
    alpha = 10  # damping / decay rate

    def resonator_filter(signal, fs, f0, alpha):
        x = np.asarray(signal, dtype=np.float64)
        if x.ndim > 1:
            x = x[:, 0]
        y = np.empty_like(x, dtype=np.float64)
        Z = np.exp(-alpha / fs) * np.exp(1j * 2 * np.pi * f0 / fs)
        z_n = 0j
        for i, u in enumerate(x):
            z_n = Z * z_n + u
            y[i] = np.real(z_n)
        return y

    def freqs(N, base_freq, spacing):
        f = base_freq
        while f <= 20000:
            yield f
            f *= spacing
    
    duration_seconds = 10.0
    num_samples = int(fs * duration_seconds)
    signal = np.zeros(num_samples, dtype=np.float64)

    for d in np.arange(0, duration_seconds, 1):
        index = int(d * fs)
        if index < num_samples:
            signal[index] = 1.0

    N = 50
    base_freq = 220.0
    octave_spacing = 1.4

    filter_freq = np.array(list(freqs(N, base_freq, octave_spacing)))
    filtered_outputs = [resonator_filter(signal, fs, f, alpha) for f in filter_freq]

    summe = np.sum(filtered_outputs, axis=0)
    summe = summe / max(1, len(filtered_outputs))

    print('Bank output')
    print(filter_freq)
    Audio(summe, rate=int(fs))

Bank output
[  220.           308.           431.2          603.68
   845.152       1183.2128      1656.49792     2319.097088
  3246.7359232   4545.43029248  6363.60240947  8909.04337326
 12472.66072257 17461.72501159]


In [24]:

print(sd.query_devices())
sd.default.device = 3
def callback(outdata, frames, time, status):

    if status:
        print(status)
    
    outdata[:]
with sd.OutputStream(channels=1, callback=callback, samplerate=int(fs)):
    sd.sleep(int(duration_seconds * 1000))




   0 VG248, Core Audio (0 in, 2 out)
   1 Mikrofon von „iPhone“, Core Audio (1 in, 0 out)
   2 MacBook Pro-Mikrofon, Core Audio (1 in, 0 out)
*  3 MacBook Pro-Lautsprecher, Core Audio (0 in, 2 out)
   4 Microsoft Teams Audio, Core Audio (1 in, 1 out)
   5 Pro Tools Audio Bridge 16, Core Audio (16 in, 16 out)
   6 Pro Tools Audio Bridge 2-A, Core Audio (2 in, 2 out)
   7 Pro Tools Audio Bridge 2-B, Core Audio (2 in, 2 out)
   8 Pro Tools Audio Bridge 32, Core Audio (32 in, 32 out)
   9 Pro Tools Audio Bridge 64, Core Audio (64 in, 64 out)
  10 Pro Tools Audio Bridge 6, Core Audio (6 in, 6 out)
  11 Pro Tools Aggregate I/O, Core Audio (1 in, 2 out)
  12 rekordbox Aggregate Device, Core Audio (0 in, 2 out)
||PaMacCore (AUHAL)|| Warning on line 521: err=''!obj'', msg=Unknown Error
||PaMacCore (AUHAL)|| Warning on line 441: err=''!obj'', msg=Unknown Error
||PaMacCore (AUHAL)|| Error on line 1332: err='-10851', msg=Audio Unit: Invalid Property Value


PortAudioError: Error opening OutputStream: Internal PortAudio error [PaErrorCode -9986]

In [21]:
fs = 48000.0
alpha = 5
base_freq = 220.0
octave_spacing = 1.7

def freqs(base_freq, spacing):
    f = base_freq
    while f <= 20000:
        yield f
        f *= spacing

filter_freqs = list(freqs(base_freq, octave_spacing))


Z = [np.exp(-alpha/fs) * np.exp(1j * 2*np.pi*f/fs) for f in filter_freqs]
states = [0j for _ in filter_freqs]

def callback(outdata, frames, time, status):
    if status:
        print(status)

    # output buffer
    out = np.zeros(frames, dtype=np.float64)

    for i in range(frames):
        # impulse every second (example)
        u = 1.0 if int(callback.pos) % int(fs) == 0 else 0.0

        s = 0.0
        for k in range(len(states)):
            states[k] = Z[k] * states[k] + u
            s += np.real(states[k])

        out[i] = s
        callback.pos += 1

    # normalize and write to output
    outdata[:] = (out / len(states)).reshape(-1, 1)

callback.pos = 0

sd.default.device = 3  # or your duplex device
with sd.OutputStream(channels=1, samplerate=int(fs), callback=callback):
    sd.sleep(10000)  # 10 seconds


In [22]:
params = {
    "alpha": 10,
    "base_freq": 220.0,
    "spacing": 1.4
}


alpha_slider = widgets.FloatSlider(
    value=params["alpha"],
    min=0.1,
    max=50,
    step=0.1,
    description='alpha'
)

base_freq_slider = widgets.FloatSlider(
    value=params["base_freq"],
    min=20,
    max=2000,
    step=1,
    description='base_freq'
)

spacing_slider = widgets.FloatSlider(
    value=params["spacing"],
    min=1.01,
    max=2.0,
    step=0.01,
    description='spacing'
)

def update_alpha(change):
    params["alpha"] = change["new"]

def update_base_freq(change):
    params["base_freq"] = change["new"]

def update_spacing(change):
    params["spacing"] = change["new"]

alpha_slider.observe(update_alpha, names='value')
base_freq_slider.observe(update_base_freq, names='value')
spacing_slider.observe(update_spacing, names='value')

alpha_slider.observe(update_alpha, names='value')
base_freq_slider.observe(update_base_freq, names='value')
spacing_slider.observe(update_spacing, names='value')

display(alpha_slider)
display(base_freq_slider)
display(spacing_slider)



def freqs(base_freq, spacing):
    f = base_freq
    while f <= 20000:
        yield f
        f *= spacing

fs = 48000.0

# initial frequencies
filter_freqs = list(freqs(params["base_freq"], params["spacing"]))

# resonator coefficients and states
Z = [np.exp(-params["alpha"]/fs) * np.exp(1j * 2*np.pi*f/fs) for f in filter_freqs]
states = [0j for _ in filter_freqs]


last_params = params.copy()

def callback(outdata, frames, time, status):
    if status:
        print(status)

    global filter_freqs, Z, states, last_params

    # Check if ANY parameter changed
    if params != last_params:
        last_params = params.copy()

        # Recompute frequencies
        filter_freqs = list(freqs(params["base_freq"], params["spacing"]))

        # Recompute Z coefficients
        Z = [np.exp(-params["alpha"]/fs) * np.exp(1j * 2*np.pi*f/fs)
             for f in filter_freqs]

        # Reset states (or keep them if you prefer)
        states = [0j for _ in filter_freqs]

    out = np.zeros(frames)

    for i in range(frames):
        u = 1.0 if int(callback.pos) % int(fs) == 0 else 0.0

        s = 0.0
        for k in range(len(states)):
            states[k] = Z[k] * states[k] + u
            s += np.real(states[k])

        out[i] = s
        callback.pos += 1

    outdata[:] = (out / len(states)).reshape(-1, 1)

callback.pos = 0


with sd.OutputStream(channels=1, samplerate=int(fs), callback=callback):
    sd.sleep(10000)  # keep running


FloatSlider(value=10.0, description='alpha', max=50.0, min=0.1)

FloatSlider(value=220.0, description='base_freq', max=2000.0, min=20.0, step=1.0)

FloatSlider(value=1.4, description='spacing', max=2.0, min=1.01, step=0.01)